# Week 7: Hello GIS!

Welcome to Python for GIS. This is your first notebook—a document that mixes explanations with runnable code.

**What you'll do:**
1. Set up your environment
2. Learn how data paths work (important for future weeks!)
3. Create your first map with GeoPandas

**How to run cells:** Click on a code cell and press `Shift + Enter`

---

## Where is this notebook running?

This is an important concept for the Python weeks:

| If you opened with... | The notebook runs on... | Your files live on... |
|----------------------|-------------------------|----------------------|
| **Google Colab** | A Google computer in a data center | Your Google Drive |
| **Jupyter (local)** | Your own computer | Your computer's hard drive |

**Why does this matter?**

When you write `gpd.read_file("data/my_file.geojson")`, the code looks for that file wherever it's running. In Colab, that's a Google server—which can't see your laptop's files!

**The solution:** In Colab, we "mount" (connect) your Google Drive so the code can access files you've stored there. You'll learn this in the next cell.

---

## Step 1: Set up your environment

Run this cell first. It detects where you are and installs packages if needed.

In [ ]:
# Detect environment
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("You're running in Google Colab")
    print("Installing GIS packages (takes ~1 minute)...")
    !pip install geopandas contextily folium -q
    print("Done!")
else:
    print("You're running locally (Jupyter)")
    print("Make sure you activated your environment: conda activate intro-gis")

---

## Step 2: Set up Google Drive (Colab users only)

**What is "mounting" a drive?**

When you plug a USB stick into your computer, it "mounts"—suddenly your computer can see the files on it. Mounting Google Drive in Colab works the same way: it connects your Drive to the Colab computer so your code can read your files.

**For future weeks**, you'll store your data files in Google Drive. This week we'll just set up the connection and create your folder structure.

Run the cell below. If you're in Colab, it will:
1. Ask permission to access your Drive (click "Connect")
2. Create your `intro-gis` folder structure

In [ ]:
from pathlib import Path

if IN_COLAB:
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Create folder structure for future weeks
    base = Path("/content/drive/MyDrive/intro-gis")
    folders = ["data/raw", "data/processed", "outputs"]
    
    for folder in folders:
        (base / folder).mkdir(parents=True, exist_ok=True)
    
    print(f"Created folder structure in Google Drive:")
    print(f"  {base}/")
    for folder in folders:
        print(f"    {folder}/")
    print("\nFolder structure explained:")
    print("  - data/raw/      : Store your original data files here (never modify)")
    print("  - data/processed/: Your code will save analysis results here")
    print("  - outputs/       : Save final maps and reports here")
    print("\nYou can now upload data files to data/raw/ in Google Drive!")
else:
    print("Local Jupyter: Your files should be in your intro-gis folder.")
    print("Folder structure:")
    print("  - data/raw/      : Store your original data files here (never modify)")
    print("  - data/processed/: Your code will save analysis results here")
    print("  - outputs/       : Save final maps and reports here")
    print("\nNo Drive mounting needed.")

---

## Step 3: Import libraries

Libraries are collections of code that other people wrote. Instead of writing mapping code from scratch, we use `geopandas`.

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import Point

print("Libraries imported!")

---

## Step 3b: Set up data paths

**Important concept for all future weeks!**

We use two main folders:
- **RAW**: Where you store original data files (read-only, never modify these)
- **PROCESSED**: Where your code saves analysis results

This keeps your original data safe and organized.

In [ ]:
# Set up data paths
if IN_COLAB:
    BASE = Path("/content/drive/MyDrive/intro-gis")
else:
    BASE = Path("..")  # One level up from notebooks folder

RAW = BASE / "data" / "raw"
PROCESSED = BASE / "data" / "processed"
OUTPUTS = BASE / "outputs"

print("Data paths configured:")
print(f"  RAW (input):  {RAW}")
print(f"  PROCESSED:    {PROCESSED}")
print(f"  OUTPUTS:      {OUTPUTS}")
print("\nExample usage:")
print(f"  Load data:  gpd.read_file(RAW / 'my_data.geojson')")
print(f"  Save data:  df.to_file(PROCESSED / 'results.geojson')")
print(f"  Save plot:  plt.savefig(OUTPUTS / 'my_map.png')")

---

## Step 4: Create sample data

This week we'll create data directly in Python (no files needed). In future weeks, you'll load files from your Drive.

In [ ]:
# Australian capital cities
cities_data = {
    'city': ['Sydney', 'Melbourne', 'Brisbane', 'Perth', 'Adelaide'],
    'population': [5312000, 5078000, 2514000, 2085000, 1376000],
    'longitude': [151.2093, 144.9631, 153.0251, 115.8605, 138.6007],
    'latitude': [-33.8688, -37.8136, -27.4698, -31.9505, -34.9285]
}

# Convert to a GeoDataFrame (like a shapefile in memory)
df = pd.DataFrame(cities_data)
geometry = [Point(xy) for xy in zip(df['longitude'], df['latitude'])]
cities = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')

print(f"Created GeoDataFrame with {len(cities)} cities")
cities

---

## Step 5: Load data from a URL

GeoPandas can load data directly from the internet. Here we'll load country boundaries from Natural Earth.

In [ ]:
# Load world countries from Natural Earth (this takes ~10 seconds)
print("Loading world boundaries from Natural Earth...")
world = gpd.read_file("https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip")

# Filter to just Australia
australia = world[world['NAME'] == 'Australia']

print(f"Loaded {len(world)} countries")
print(f"Filtered to: {australia['NAME'].values[0]}")

---

## Step 6: Create your first map!

This is equivalent to adding layers and styling them in QGIS.

In [ ]:
# Create figure
fig, ax = plt.subplots(figsize=(12, 8))

# Plot Australia boundary (like adding a polygon layer)
australia.plot(ax=ax, color='lightgreen', edgecolor='darkgreen', linewidth=1)

# Plot cities (like adding a point layer)
# Size based on population
cities.plot(ax=ax, color='red', markersize=cities['population']/15000, zorder=5)

# Add labels (like enabling labels in QGIS)
for idx, row in cities.iterrows():
    ax.annotate(row['city'], 
                xy=(row['longitude'], row['latitude']),
                xytext=(5, 5), 
                textcoords='offset points', 
                fontsize=10)

# Style the map
ax.set_title('Australian Capital Cities', fontsize=14)
ax.set_xlim(110, 160)
ax.set_ylim(-45, -10)
ax.set_facecolor('lightblue')

plt.show()

# Save the map to outputs folder
output_path = OUTPUTS / "australian_cities_map.png"
fig.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Your first Python map!")
print(f"Map saved to: {output_path}")

---

## Step 7: Save your data

**Good practice:** Save your processed data so you can use it later without re-running analysis.

This is especially useful when:
- Your analysis takes a long time
- You want to use the data in other notebooks
- You want to share results with others

In [ ]:
# Save the cities GeoDataFrame to the PROCESSED folder
cities_output = PROCESSED / "australian_cities.geojson"
cities.to_file(cities_output, driver='GeoJSON')
print(f"Cities data saved to: {cities_output}")

# Save the Australia boundary too
australia_output = PROCESSED / "australia_boundary.geojson"
australia.to_file(australia_output, driver='GeoJSON')
print(f"Australia boundary saved to: {australia_output}")

print("\nThese files can now be loaded in future notebooks using:")
print(f"  cities = gpd.read_file(PROCESSED / 'australian_cities.geojson')")
print(f"  australia = gpd.read_file(PROCESSED / 'australia_boundary.geojson')")

---

## Try it yourself

**Exercise:** Add Darwin and Hobart to the cities data:
- Darwin: longitude 130.8456, latitude -12.4634, population 147000
- Hobart: longitude 147.3272, latitude -42.8821, population 238000

Then re-run the map cell above.

In [ ]:
# Your code here - add Darwin and Hobart
# Hint: You can add rows to the cities_data dictionary and re-run from Step 4


---

## Summary: What you learned

1. **Colab vs Local:** Notebooks can run on Google's servers (Colab) or your computer (Jupyter)
2. **Mounting Drive:** Connects your Google Drive to Colab so your code can access your files
3. **Folder structure:** 
   - `data/raw/` - Store original data files (never modify)
   - `data/processed/` - Save analysis results
   - `outputs/` - Save final maps and reports
4. **Data paths:** Use `RAW`, `PROCESSED`, and `OUTPUTS` variables to keep code organized
5. **Loading data:** `gpd.read_file()` loads spatial data from files or URLs
6. **Creating maps:** `.plot()` creates visualizations (like styling layers in QGIS)
7. **Saving outputs:** 
   - Data: `gdf.to_file(PROCESSED / 'filename.geojson')`
   - Maps: `fig.savefig(OUTPUTS / 'map.png')`

**Next week:** You'll load real data files from your RAW folder and perform spatial analysis!

---

**Save your work:**
- Colab: `File > Save a copy in Drive`
- Local: `Ctrl+S` or `Cmd+S`